In [ ]:
# In[1]:

import pandas as pd
import numpy as np

# Load CSVs (they are in the current working directory)
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_error = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes (Unix seconds -> UTC)
for df in (df_metric, df_trace, df_log, df_error):
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# 1) metric.csv aggregation
metric_group = df_metric.groupby(['cmdb_id', 'kpi_name'], dropna=False)
metric_agg = metric_group.agg(
    count=('value', 'size'),
    min_timestamp=('timestamp', 'min'),
    max_timestamp=('timestamp', 'max'),
    p95_value=('value', lambda s: float(np.percentile(s.dropna(), 95)) if len(s.dropna())>0 else np.nan)
).reset_index().sort_values('count', ascending=False)
metric_top = metric_agg[['cmdb_id','kpi_name','count','min_timestamp','max_timestamp','p95_value']].head(20)

# 2) trace.csv aggregation
trace_group = df_trace.groupby(['cmdb_id', 'trace_name'], dropna=False)
trace_agg = trace_group.agg(
    count=('value', 'size'),
    min_timestamp=('timestamp', 'min'),
    max_timestamp=('timestamp', 'max'),
    p95_value=('value', lambda s: float(np.percentile(s.dropna(), 95)) if len(s.dropna())>0 else np.nan)
).reset_index().sort_values('count', ascending=False)
trace_top = trace_agg[['cmdb_id','trace_name','count','min_timestamp','max_timestamp','p95_value']].head(20)

# 3) log.csv aggregation
log_group = df_log.groupby(['cmdb_id', 'log_name'], dropna=False)
log_agg = log_group.agg(
    count=('value', 'size'),
    min_timestamp=('timestamp', 'min'),
    max_timestamp=('timestamp', 'max'),
    p95_value=('value', lambda s: float(np.percentile(s.dropna(), 95)) if len(s.dropna())>0 else np.nan)
).reset_index().sort_values('count', ascending=False)
log_top = log_agg[['cmdb_id','log_name','count','min_timestamp','max_timestamp','p95_value']].head(20)

# 4) error_logs.csv aggregation: per cmdb_id count, earliest/latest timestamp, and up to 5 sample messages (truncated)
def sample_messages(msg_series, max_samples=5, max_len=200):
    # take first max_samples messages (preserve order), truncate each, join with " || "
    samples = msg_series.dropna().astype(str).head(max_samples).tolist()
    samples = [ (m[:max_len] + '...') if len(m)>max_len else m for m in samples ]
    return " || ".join(samples)

error_group = df_error.groupby('cmdb_id', dropna=False)
error_agg = error_group.agg(
    count=('message', 'size'),
    min_timestamp=('timestamp', 'min'),
    max_timestamp=('timestamp', 'max'),
).reset_index()

# add sample messages column
sample_msgs = error_group['message'].apply(lambda s: sample_messages(s, max_samples=5, max_len=200)).reset_index(name='sample_messages')
error_agg = error_agg.merge(sample_msgs, on='cmdb_id', how='left')

error_top = error_agg.sort_values('count', ascending=False)[['cmdb_id','count','min_timestamp','max_timestamp','sample_messages']].head(20)

# Display compact results (top 20 rows each)
metric_top, trace_top, log_top, error_top

```
Out[1]:
```
```python
summary = (
    "Summary of telemetry aggregates:\n"
    "- Metrics: the 'carts' service shows the largest number of KPI series and troubling P95s: latency-90 P95 ≈ 7.02s, latency-50 P95 ≈ 1.90s, high socket P95 (~61.7) and very large memory values. 'carts-db' also shows high resource P95s (CPU ~44.2, disk I/O ~3.32e6, memory ~1.45e8).\n"
    "- Traces: trace.csv returned no rows (no trace features available).\n"
    "- Logs: error counts are effectively zero across services; row_count is highest for front-end, user, and carts-db. 'carts' log series has slightly fewer rows (21) than many others (25).\n"
    "- Error messages: error_logs.csv had no aggregated rows (no sample error messages collected).\n\n"
    "Conclusion: the telemetry points to the 'carts' service (and its database 'carts-db') as the primary area of concern — high request latency on carts with elevated socket and memory figures, and carts-db showing high disk I/O and CPU. Investigate carts service request handling and the carts-db disk/CPU performance as the likely root cause."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id    kpi_name  count             min_timestamp             max_timestamp     p95_value
0          carts         cpu     25 2024-01-19 07:11:00+00:00 2024-01-19 07:35:00+00:00  1.356383e+00
1          carts       error     25 2024-01-19 07:11:00+00:00 2024-01-19 07:35:00+00:00  0.000000e+00
2          carts  latency-50     25 2024-01-19 07:11:00+00:00 2024-01-19 07:35:00+00:00  1.901469e+00
3          carts  latency-90     25 2024-01-19 07:11:00+00:00 2024-01-19 07:35:00+00:00  7.019657e+00
4          carts         mem     25 2024-01-19 07:11:00+00:00 2024-01-19 07:35:00+00:00  2.232384e+08
5          carts      socket     25 2024-01-19 07:11:00+00:00 2024-01-19 07:35:00+00:00  6.172333e+01
6          carts    workload     25 2024-01-19 07:11:00+00:00 2024-01-19 07:35:00+00:00  7.472281e+00
7       carts-db         cpu     25 2024-01-19 07:11:00+00:00 2024-01-19 07:35:00+00:00  4.415695e+01
8       carts-db      diskio     25 2024-01-19 07:11:00+00:00 2024-01-19 07:35:00+00:00  3.322482e+06
9       carts-db         mem     25 2024-01-19 07:11:00+00:00 2024-01-19 07:35:00+00:00  1.442577e+08
10      carts-db      socket     25 2024-01-19 07:11:00+00:00 2024-01-19 07:35:00+00:00  3.500000e+01
11     catalogue         cpu     25 2024-01-19 07:11:00+00:00 2024-01-19 07:35:00+00:00  1.644121e-01
12     catalogue      diskio     25 2024-01-19 07:11:00+00:00 2024-01-19 07:35:00+00:00  0.000000e+00
13     catalogue       error     25 2024-01-19 07:11:00+00:00 2024-01-19 07:35:00+00:00  0.000000e+00
14     catalogue  latency-50     25 2024-01-19 07:11:00+00:00 2024-01-19 07:35:00+00:00  3.111318e-03
15     catalogue  latency-90     25 2024-01-19 07:11:00+00:00 2024-01-19 07:35:00+00:00  4.800373e-03
16     catalogue         mem     25 2024-01-19 07:11:00+00:00 2024-01-19 07:35:00+00:00  6.498509e+06
17     catalogue      socket     25 2024-01-19 07:11:00+00:00 2024-01-19 07:35:00+00:00  8.000000e+00
18     catalogue    workload     25 2024-01-19 07:11:00+00:00 2024-01-19 07:35:00+00:00  3.728913e+00
19  catalogue-db         cpu     25 2024-01-19 07:11:00+00:00 2024-01-19 07:35:00+00:00  1.939801e-01, Empty DataFrame
Columns: [cmdb_id, trace_name, count, min_timestamp, max_timestamp, p95_value]
Index: [],          cmdb_id         log_name  count             min_timestamp             max_timestamp  p95_value
3       carts-db    log.row_count     25 2024-01-19 07:11:00+00:00 2024-01-19 07:35:00+00:00      531.2
2       carts-db  log.error_count     25 2024-01-19 07:11:00+00:00 2024-01-19 07:35:00+00:00        0.0
5      catalogue    log.row_count     25 2024-01-19 07:11:00+00:00 2024-01-19 07:35:00+00:00      149.8
4      catalogue  log.error_count     25 2024-01-19 07:11:00+00:00 2024-01-19 07:35:00+00:00        0.0
6      front-end  log.error_count     25 2024-01-19 07:11:00+00:00 2024-01-19 07:35:00+00:00        0.0
7      front-end    log.row_count     25 2024-01-19 07:11:00+00:00 2024-01-19 07:35:00+00:00     1768.0
9         orders    log.row_count     25 2024-01-19 07:11:00+00:00 2024-01-19 07:35:00+00:00      116.0
8         orders  log.error_count     25 2024-01-19 07:11:00+00:00 2024-01-19 07:35:00+00:00        0.0
21          user    log.row_count     25 2024-01-19 07:11:00+00:00 2024-01-19 07:35:00+00:00      546.4
20          user  log.error_count     25 2024-01-19 07:11:00+00:00 2024-01-19 07:35:00+00:00        0.0
13       payment    log.row_count     25 2024-01-19 07:11:00+00:00 2024-01-19 07:35:00+00:00       98.0
12       payment  log.error_count     25 2024-01-19 07:11:00+00:00 2024-01-19 07:35:00+00:00        0.0
14  queue-master  log.error_count     25 2024-01-19 07:11:00+00:00 2024-01-19 07:35:00+00:00        0.0
15  queue-master    log.row_count     25 2024-01-19 07:11:00+00:00 2024-01-19 07:35:00+00:00      522.0
18      shipping  log.error_count     25 2024-01-19 07:11:00+00:00 2024-01-19 07:35:00+00:00        0.0
19      shipping    log.row_count     25 2024-01-19 07:11:00+00:00 2024-01-19 07:35:00+00:00       58.0
1          carts    log.row_count     21 2024-01-19 07:12:00+00:00 2024-01-19 07:35:00+00:00      100.0
0          carts  log.error_count     21 2024-01-19 07:12:00+00:00 2024-01-19 07:35:00+00:00        0.0
16    session-db  log.error_count      5 2024-01-19 07:12:00+00:00 2024-01-19 07:32:00+00:00        0.0
17    session-db    log.row_count      5 2024-01-19 07:12:00+00:00 2024-01-19 07:32:00+00:00        5.0, Empty DataFrame
Columns: [cmdb_id, count, min_timestamp, max_timestamp, sample_messages]
Index: [])```
```

In [ ]:
# In[2]:

import numpy as np
import pandas as pd

# Reuse df_metric loaded previously in the session
# Filter to the requested service components
services = ["carts","catalogue","orders","payment","user"]
df_m = df_metric[df_metric['cmdb_id'].isin(services)].copy()

# 1) Compute global P95 per (cmdb_id, kpi_name) using the full series (no time filtering)
p95_df = (
    df_m.groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(lambda s: float(np.percentile(s.dropna(), 95)) if s.dropna().size > 0 else np.nan)
    .reset_index()
    .rename(columns={'value': 'global_p95'})
)

# Merge global_p95 back to the metric rows for these services
df_m = df_m.merge(p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# 2) Filter rows to the incident window (inclusive)
start = pd.to_datetime('2024-01-19 07:08:40', utc=True)
end = pd.to_datetime('2024-01-19 07:38:40', utc=True)
rows_in_window = df_m[(df_m['timestamp'] >= start) & (df_m['timestamp'] <= end)].copy()

# If no rows in window, result will be empty
if rows_in_window.empty:
    result_window = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','global_p95','count_points_in_window',
        'count_points_exceeding_p95','earliest_timestamp_exceeding_p95','max_value_in_window'
    ])
else:
    # Mark exceedances using pre-computed global_p95 (rule: threshold computed before window filtering)
    rows_in_window['exceed'] = rows_in_window['value'] >= rows_in_window['global_p95']

    # Aggregations per (cmdb_id, kpi_name)
    grp = rows_in_window.groupby(['cmdb_id', 'kpi_name'], as_index=False)
    agg = grp.agg(
        global_p95=('global_p95','first'),
        count_points_in_window=('value','size'),
        count_points_exceeding_p95=('exceed','sum'),
        max_value_in_window=('value','max')
    )

    # Compute earliest timestamp where exceed == True per group
    def earliest_exceed_timestamp(g):
        ex = g.loc[g['exceed'], 'timestamp']
        return ex.min() if not ex.empty else pd.NaT

    earliest = rows_in_window.groupby(['cmdb_id','kpi_name']).apply(earliest_exceed_timestamp).reset_index(name='earliest_timestamp_exceeding_p95')
    agg = agg.merge(earliest, on=['cmdb_id','kpi_name'], how='left')

    # Replace NaT with None for display as requested
    agg['earliest_timestamp_exceeding_p95'] = agg['earliest_timestamp_exceeding_p95'].where(agg['earliest_timestamp_exceeding_p95'].notna(), None)

    # Compute severity ratio for sorting: max_value_in_window / global_p95, handle global_p95 == 0
    def compute_ratio(row):
        gp = row['global_p95']
        mv = row['max_value_in_window']
        if pd.isna(gp) or gp == 0:
            return np.inf if (not pd.isna(mv) and mv > 0) else 0.0
        return mv / gp

    agg['severity_ratio'] = agg.apply(compute_ratio, axis=1)

    # Sort by count_points_exceeding_p95 desc, then severity_ratio desc
    agg_sorted = agg.sort_values(['count_points_exceeding_p95','severity_ratio'], ascending=[False, False])

    # Select and order requested columns, limit to top 20
    result_window = agg_sorted[
        ['cmdb_id','kpi_name','global_p95','count_points_in_window',
         'count_points_exceeding_p95','earliest_timestamp_exceeding_p95','max_value_in_window']
    ].head(20)

# Display the compact result
result_window

```
Out[2]:
```
```python
summary = (
    "Summary of metric-window analysis (services: carts, catalogue, orders, payment, user):\n\n"
    "Key notes:\n"
    "- Many series (several 'error' and 'diskio' entries) have a global P95 of 0.0, so all 25 points in the window meet the >= P95 condition; those are not informative for identifying resource spikes.\n\n"
    "Most meaningful findings (non-zero global P95):\n"
    "- catalogue.socket: global_p95=8.0, 17/25 points >= P95, earliest exceed 2024-01-19 07:11 UTC, max=8.0 — persistent socket-level signal in catalogue.\n"
    "- user.socket: global_p95=20.0, 7/25 points >= P95, earliest exceed 2024-01-19 07:11 UTC, max=20.0 — intermittent socket spikes for user.\n"
    "- carts.socket: global_p95≈61.72, 2/25 points >= P95, earliest exceed 2024-01-19 07:22 UTC, max≈65.67 — brief high socket values on carts.\n"
    "- carts latency-90: global_p95≈7.02, 2/25 points >= P95, earliest exceed 2024-01-19 07:16 UTC, max≈7.20 — brief latency spikes on carts.\n"
    "- orders.cpu: global_p95≈4.82, 2/25 points >= P95, earliest exceed 2024-01-19 07:18 UTC, max≈7.21 — short CPU spike on orders.\n"
    "- Several other latency/workload/cpu series across payment, user, catalogue, and orders show 1–2 exceedances each in the window (brief spikes).\n\n"
    "Conclusion / recommended next steps:\n"
    "- The strongest sustained signal is catalogue.socket (17/25 exceedances) — investigate socket/network issues for the catalogue service first.\n"
    "- Also investigate socket behavior for user and carts, and short latency spikes in carts and CPU spikes in orders.\n"
    "- Treat the zero-P95 series (errors/diskio) with caution: their \"exceed\" counts are driven by zero thresholds and need separate interpretation.\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  global_p95  count_points_in_window  count_points_exceeding_p95 earliest_timestamp_exceeding_p95  max_value_in_window
1       carts       error    0.000000                      25                          25        2024-01-19 07:11:00+00:00             0.000000
8   catalogue      diskio    0.000000                      25                          25        2024-01-19 07:11:00+00:00             0.000000
9   catalogue       error    0.000000                      25                          25        2024-01-19 07:11:00+00:00             0.000000
16     orders       error    0.000000                      25                          25        2024-01-19 07:11:00+00:00             0.000000
23    payment      diskio    0.000000                      25                          25        2024-01-19 07:11:00+00:00             0.000000
24    payment       error    0.000000                      25                          25        2024-01-19 07:11:00+00:00             0.000000
31       user       error    0.000000                      25                          25        2024-01-19 07:11:00+00:00             0.000000
13  catalogue      socket    8.000000                      25                          17        2024-01-19 07:11:00+00:00             8.000000
35       user      socket   20.000000                      25                           7        2024-01-19 07:11:00+00:00            20.000000
15     orders         cpu    4.821688                      25                           2        2024-01-19 07:18:00+00:00             7.214619
26    payment  latency-90    0.248787                      25                           2        2024-01-19 07:30:00+00:00             0.288769
17     orders  latency-50    0.870406                      25                           2        2024-01-19 07:16:00+00:00             0.984586
25    payment  latency-50    0.074118                      25                           2        2024-01-19 07:29:00+00:00             0.080909
5       carts      socket   61.723333                      25                           2        2024-01-19 07:22:00+00:00            65.666667
2       carts  latency-50    1.901469                      25                           2        2024-01-19 07:16:00+00:00             2.022946
30       user         cpu    0.926373                      25                           2        2024-01-19 07:18:00+00:00             0.974015
29    payment    workload    1.904663                      25                           2        2024-01-19 07:13:00+00:00             1.974450
7   catalogue         cpu    0.164412                      25                           2        2024-01-19 07:18:00+00:00             0.170139
32       user  latency-50    0.004866                      25                           2        2024-01-19 07:11:00+00:00             0.004994
3       carts  latency-90    7.019657                      25                           2        2024-01-19 07:16:00+00:00             7.203227```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd

# Reuse df_metric from session
services = ["carts","catalogue","orders","payment","user"]
df_m = df_metric[df_metric['cmdb_id'].isin(services)].copy()

# 1) Global P95 per series (full series)
p95_df = (
    df_m.groupby(['cmdb_id','kpi_name'], dropna=False)['value']
      .agg(lambda s: float(np.percentile(s.dropna(),95)) if s.dropna().size>0 else np.nan)
      .reset_index()
      .rename(columns={'value':'global_p95'})
)

df_m = df_m.merge(p95_df, on=['cmdb_id','kpi_name'], how='left')

# 2) Filter to incident window (inclusive)
start = pd.to_datetime('2024-01-19 07:08:40', utc=True)
end   = pd.to_datetime('2024-01-19 07:38:40', utc=True)
window_rows = df_m[(df_m['timestamp'] >= start) & (df_m['timestamp'] <= end)].copy()

# Exclude series with global_p95 == 0 or NaN
window_rows = window_rows[window_rows['global_p95'].notna() & (window_rows['global_p95'] > 0)].copy()

# If no candidate rows, return empty DataFrame with specified columns
out_cols = ['cmdb_id','kpi_name','run_start_timestamp','run_end_timestamp',
            'run_length_points','max_value_in_run','global_p95','severity_ratio']

if window_rows.empty:
    runs_top = pd.DataFrame(columns=out_cols)
else:
    window_rows = window_rows.sort_values(['cmdb_id','kpi_name','timestamp']).copy()
    # Identify consecutive segments per group by iterating groups to avoid index alignment issues
    pieces = []
    for (cmdb, kpi), g in window_rows.groupby(['cmdb_id','kpi_name'], sort=False):
        gg = g.copy()
        gg['exceed'] = gg['value'] >= gg['global_p95']
        # segment id increments when exceed changes (True/False)
        gg['segment'] = (gg['exceed'] != gg['exceed'].shift()).cumsum()
        pieces.append(gg)
    window_seg = pd.concat(pieces, ignore_index=True)

    # Select only segments where exceed == True
    exceed_rows = window_seg[window_seg['exceed']].copy()
    if exceed_rows.empty:
        runs_top = pd.DataFrame(columns=out_cols)
    else:
        runs = (
            exceed_rows
            .groupby(['cmdb_id','kpi_name','segment'], sort=False, as_index=False)
            .agg(
                run_start_timestamp=('timestamp','min'),
                run_end_timestamp=('timestamp','max'),
                run_length_points=('value','size'),
                max_value_in_run=('value','max'),
                global_p95=('global_p95','first')
            )
        )
        runs['severity_ratio'] = runs['max_value_in_run'] / runs['global_p95']
        runs_top = (
            runs.sort_values(['run_length_points','severity_ratio'], ascending=[False, False])
                [ ['cmdb_id','kpi_name','run_start_timestamp','run_end_timestamp',
                   'run_length_points','max_value_in_run','global_p95','severity_ratio'] ]
                .head(20)
                .reset_index(drop=True)
        )

# Display compact result (top runs)
runs_top

```
Out[3]:
```
```python
summary = (
    "Summary of consecutive exceedance runs (UTC):\n\n"
    "- Strongest sustained run: catalogue.socket — 17 consecutive points from 2024-01-19 07:11 to 2024-01-19 07:27 UTC (severity_ratio=1.0). This is the clearest persistent signal.\n"
    "- Secondary sustained run: user.socket — 7 consecutive points from 2024-01-19 07:11 to 2024-01-19 07:17 UTC (severity_ratio=1.0).\n"
    "- Notable short/high-severity runs:\n"
    "  * orders.cpu — 2 points (07:18–07:19 UTC), severity_ratio≈1.50 (largest relative spike).\n"
    "  * carts.socket — 2 points (07:22–07:23 UTC), severity_ratio≈1.06.\n"
    "- Several brief 1-point runs across services for latency, cpu, mem, and workload with modest severity ratios (≈1.02–1.16).\n\n"
    "Conclusion / recommended actions:\n"
    "- Prioritize investigating catalogue.socket (persistent 17-point exceedance) for socket/network issues on the catalogue service.\n"
    "- Also investigate user.socket and transient spikes in orders.cpu and carts.socket/latency.\n"
    "- Other single-point exceedances are lower priority but worth checking if they correlate with user complaints or other alerts.\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name       run_start_timestamp         run_end_timestamp  run_length_points  max_value_in_run    global_p95  severity_ratio
0   catalogue      socket 2024-01-19 07:11:00+00:00 2024-01-19 07:27:00+00:00                 17      8.000000e+00  8.000000e+00        1.000000
1        user      socket 2024-01-19 07:11:00+00:00 2024-01-19 07:17:00+00:00                  7      2.000000e+01  2.000000e+01        1.000000
2      orders         cpu 2024-01-19 07:18:00+00:00 2024-01-19 07:19:00+00:00                  2      7.214619e+00  4.821688e+00        1.496285
3       carts      socket 2024-01-19 07:22:00+00:00 2024-01-19 07:23:00+00:00                  2      6.566667e+01  6.172333e+01        1.063887
4      orders      socket 2024-01-19 07:34:00+00:00 2024-01-19 07:35:00+00:00                  2      2.521951e+01  2.486333e+01        1.014325
5      orders         mem 2024-01-19 07:18:00+00:00 2024-01-19 07:19:00+00:00                  2      3.318126e+08  3.312225e+08        1.001782
6       carts         mem 2024-01-19 07:28:00+00:00 2024-01-19 07:29:00+00:00                  2      2.232489e+08  2.232384e+08        1.000047
7     payment  latency-90 2024-01-19 07:30:00+00:00 2024-01-19 07:30:00+00:00                  1      2.887688e-01  2.487867e-01        1.160709
8      orders  latency-50 2024-01-19 07:16:00+00:00 2024-01-19 07:16:00+00:00                  1      9.845858e-01  8.704057e-01        1.131180
9     payment  latency-50 2024-01-19 07:35:00+00:00 2024-01-19 07:35:00+00:00                  1      8.090857e-02  7.411807e-02        1.091617
10      carts  latency-50 2024-01-19 07:34:00+00:00 2024-01-19 07:34:00+00:00                  1      2.022946e+00  1.901469e+00        1.063886
11       user         cpu 2024-01-19 07:32:00+00:00 2024-01-19 07:32:00+00:00                  1      9.740148e-01  9.263733e-01        1.051428
12    payment    workload 2024-01-19 07:19:00+00:00 2024-01-19 07:19:00+00:00                  1      1.974450e+00  1.904663e+00        1.036640
13  catalogue         cpu 2024-01-19 07:35:00+00:00 2024-01-19 07:35:00+00:00                  1      1.701386e-01  1.644121e-01        1.034831
14       user  latency-50 2024-01-19 07:11:00+00:00 2024-01-19 07:11:00+00:00                  1      4.993914e-03  4.865846e-03        1.026320
15      carts  latency-90 2024-01-19 07:16:00+00:00 2024-01-19 07:16:00+00:00                  1      7.203227e+00  7.019657e+00        1.026151
16    payment      socket 2024-01-19 07:35:00+00:00 2024-01-19 07:35:00+00:00                  1      2.224390e+01  2.169667e+01        1.025222
17       user    workload 2024-01-19 07:22:00+00:00 2024-01-19 07:22:00+00:00                  1      1.719690e+01  1.680086e+01        1.023572
18      carts         cpu 2024-01-19 07:32:00+00:00 2024-01-19 07:32:00+00:00                  1      1.387458e+00  1.356383e+00        1.022910
19      carts  latency-50 2024-01-19 07:16:00+00:00 2024-01-19 07:16:00+00:00                  1      1.942302e+00  1.901469e+00        1.021474```
```